# JEV-1.13 Q-ALT hallucination detection (BenMedHallu)

Jev (`typesafe/jev-1.13`) is a **decision model**: it is called on OpenRouter's Decisions API (`/api/alpha/decisions`), not `/chat/completions`, and returns a probability for each option instead of text.

**Logic (same as the original Q-ALT notebook):** each row's question (`hallucinated_question` if present, else `question`) is shown with options A-D and E (`উত্তর নেই`). Expected label is always `1`: Jev picking **E** = successful detection (`1`), picking A-D = `0`. Accuracy = E detections / successfully scored rows x 100.

**Run order**
1. Add `OPENROUTER_API_KEY` in Colab Secrets (key icon, left sidebar) and allow notebook access. Upload `bangla_med_qa_model_evaluations.csv` to the Colab files area.
2. Run cells 1-3. Cell 3 shows a cost estimate **without any API call**.
4. Cell 5 is a 5-row smoke test (a few cents at most) that also gives a pilot-based cost projection.
5. Cell 6 asks you to type `YES` before the full run. It resumes from `jev_qalt_evaluations.csv`, skips rows already scored, and retries only FAILED/PENDING rows.
6. Cell 7 prints the final summary.

**Threshold note:** a row counts as detected when `P(E) >= THRESHOLD` (default 0.5). Because every row's expected label is `1`, this accuracy is a recall-style number; lowering the threshold can only raise it and says nothing about false alarms, so do not tune the threshold on this file.

In [1]:
# ==========================================
# 1. CONFIG
# ==========================================
import os, json, math, time, threading
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

try:
    from google.colab import userdata
    API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:
    API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found. Add it in Colab Secrets and enable notebook access.")

URL        = "https://openrouter.ai/api/alpha/decisions"   # Decisions API (NOT /chat/completions)
MODEL      = "typesafe/jev-1.13"
INPUT_FILE  = "bangla_med_qa_model_evaluations.csv"
OUTPUT_FILE = "jev_qalt_evaluations.csv"

THRESHOLD        = 0.5    # detected (label 1) if P(E) >= THRESHOLD
WORKERS          = 8      # parallel requests
MAX_ATTEMPTS     = 4      # attempts per row per run (retries on timeout / 429 / 5xx / malformed)
REQUEST_TIMEOUT  = 30     # seconds per request (Jev p50 latency is ~0.2 s)
CHECKPOINT_EVERY = 50     # rows
SMOKE_N          = 5

# Published Jev 1.13 price on OpenRouter (https://openrouter.ai/typesafe/jev-1.13, checked 2026-10-09):
# $0.042 per 1M input tokens, $0 per 1M output tokens. Used ONLY for the pre-run estimate.
# Actual cost in the results always comes from the API's own usage.cost field.
PRICE_IN_PER_M  = 0.042
PRICE_OUT_PER_M = 0.0

LABELS = ["A", "B", "C", "D", "E"]
QUESTION_KEY = "answer"
INSTRUCTIONS = (
    "You are an expert physician evaluating a multiple-choice question. "
    "Identify any factual errors, contradictory premises, or invalid clinical thresholds in the question. "
    "Choose the option that is strictly correct. If NONE of A, B, C, or D is medically correct "
    "because the question premise is flawed or hallucinated, choose E."
)
E_SUFFIX = " (none of A-D is medically correct because the question premise is flawed or hallucinated)"
DEFAULT_E_TEXT = "উত্তর নেই"


In [3]:
# ==========================================
# 2. LOAD DATA, BUILD REQUESTS, RESUME STATE (no API calls)
# ==========================================
df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} rows from {INPUT_FILE}")
if len(df) != 994:
    print(f"WARNING: expected 994 rows, found {len(df)}")
need = ["options/A", "options/B", "options/C", "options/D"]
missing_cols = [c for c in need if c not in df.columns]
if missing_cols or not ({"question", "hallucinated_question"} & set(df.columns)):
    raise ValueError(f"Missing required columns: {missing_cols or 'question / hallucinated_question'}")

def txt(v):
    return None if v is None or pd.isna(v) or str(v).strip() == "" else str(v)

def build_payload(row):
    """Returns (payload, input_error). Same question/option logic as the original notebook."""
    q = txt(row.get("hallucinated_question")) or txt(row.get("question"))
    opts = {k: txt(row.get(f"options/{k}")) for k in ["A", "B", "C", "D"]}
    if q is None:
        return None, "INPUT_ERROR: no question text"
    bad = [k for k, v in opts.items() if v is None]
    if bad:
        return None, f"INPUT_ERROR: empty option(s) {bad}"
    e_text = txt(row.get("options/E")) or DEFAULT_E_TEXT
    criteria = {**opts, "E": e_text + E_SUFFIX}
    payload = {
        "model": MODEL,
        "state": {"question": q},
        "questions": {QUESTION_KEY: {"type": "choice", "instructions": INSTRUCTIONS, "criteria": criteria}},
    }
    return payload, None

# Result columns (object dtype so strings/None/numbers can mix safely)
RES_COLS = ["jev_prob_E", "jev_choice", "jev_pred_option", "jev_probabilities", "jev_pred",
            "expected_label", "status", "error", "attempts", "input_tokens", "output_tokens",
            "total_tokens", "cost_usd", "cost_source", "model_served", "request_id"]
for c in RES_COLS:
    df[c] = pd.Series([None] * len(df), dtype=object)
df["expected_label"] = 1
df["status"] = "PENDING"
df["attempts"] = 0

# Resume from existing output (only if it matches this input)
if os.path.exists(OUTPUT_FILE):
    prev = pd.read_csv(OUTPUT_FILE)
    if len(prev) == len(df) and "status" in prev.columns:
        for c in RES_COLS:
            if c in prev.columns:
                df[c] = prev[c].astype(object).where(prev[c].notna(), None)
        df["expected_label"] = 1
        df["status"] = df["status"].fillna("PENDING")
        print(f"Resuming from {OUTPUT_FILE}: {dict(df['status'].value_counts())}")
    else:
        raise ValueError(f"{OUTPUT_FILE} exists but does not match the input ({len(prev)} vs {len(df)} rows). "
                         "Rename or delete it to start fresh.")
else:
    print("No existing output file: starting fresh.")

# Size of each request payload (used for the cost estimates)
payload_chars, payload_bytes = {}, {}
n_input_errors = 0
for i in df.index:
    p, err = build_payload(df.loc[i])
    if err:
        n_input_errors += 1
        payload_chars[i] = payload_bytes[i] = 0
    else:
        s = json.dumps(p, ensure_ascii=False)
        payload_chars[i], payload_bytes[i] = len(s), len(s.encode("utf-8"))
payload_chars, payload_bytes = pd.Series(payload_chars), pd.Series(payload_bytes)
print(f"Rows with unusable input (will be marked FAILED without an API call): {n_input_errors}")


Loaded 994 rows from bangla_med_qa_model_evaluations.csv
No existing output file: starting fresh.
Rows with unusable input (will be marked FAILED without an API call): 0


In [4]:
# ==========================================
# 3. COST ESTIMATE BEFORE ANY API CALL (offline)
# ==========================================
todo_idx = df.index[df["status"] != "OK"]
chars = int(payload_chars.loc[todo_idx].sum())
nbytes = int(payload_bytes.loc[todo_idx].sum())

# Jev's tokenizer and its handling of Bangla are not documented, so tokens cannot be predicted reliably.
# These are crude bounds on the payload only (not a forecast):
#   low  : 4 characters per token (optimistic, typical for English)
#   high : 1 token per UTF-8 byte (worst case for a byte-level tokenizer; Bangla is 3 bytes per character)
# The Decisions API may also add its own encoding of the question structure, which these bounds ignore.
lo_tok, hi_tok = chars / 4, nbytes
lo_cost, hi_cost = lo_tok / 1e6 * PRICE_IN_PER_M, hi_tok / 1e6 * PRICE_IN_PER_M

print("=" * 66)
print(f"PRE-RUN COST ESTIMATE  |  {MODEL}  |  rows still to score: {len(todo_idx)}")
print("=" * 66)
print(f"Verified price : ${PRICE_IN_PER_M}/1M input tokens, ${PRICE_OUT_PER_M}/1M output tokens")
print(f"Payload size   : {chars:,} characters, {nbytes:,} UTF-8 bytes")
print(f"Token bounds   : {lo_tok:,.0f}  to  {hi_tok:,.0f} input tokens   (UNRELIABLE, see notes)")
print(f"Cost bounds    : ${lo_cost:.4f}  to  ${hi_cost:.4f}")
print("-" * 66)
print("Token usage for Bangla text cannot be estimated reliably offline.")
print("Run cell 5 (5-row smoke test) for a pilot-based projection from real usage.")
print("Actual cost is always taken from the API's usage.cost, never from this estimate.")
print("=" * 66)


PRE-RUN COST ESTIMATE  |  typesafe/jev-1.13  |  rows still to score: 994
Verified price : $0.042/1M input tokens, $0.0/1M output tokens
Payload size   : 742,598 characters, 996,560 UTF-8 bytes
Token bounds   : 185,650  to  996,560 input tokens   (UNRELIABLE, see notes)
Cost bounds    : $0.0078  to  $0.0419
------------------------------------------------------------------
Token usage for Bangla text cannot be estimated reliably offline.
Run cell 5 (5-row smoke test) for a pilot-based projection from real usage.
Actual cost is always taken from the API's usage.cost, never from this estimate.


In [5]:
# ==========================================
# 4. ENGINE: API CALL, PARSING, RETRIES, CHECKPOINTS
# ==========================================
lock = threading.Lock()
FATAL_CODES = {401, 402, 403}          # bad key / no credits / forbidden: stop the whole run
RETRY_CODES = {408, 409, 425, 429}     # plus every 5xx

def num(x):
    return x if isinstance(x, (int, float)) and not isinstance(x, bool) and x == x else None

def parse_response(data):
    """Returns (parsed_dict, None) or (None, error_string). Never guesses a prediction."""
    if not isinstance(data, dict):
        return None, "MALFORMED: response is not a JSON object"
    if data.get("error"):
        return None, f"API_ERROR_BODY: {str(data['error'])[:200]}"
    ans = (data.get("answers") or {}).get(QUESTION_KEY)
    if not isinstance(ans, dict) or "choice" not in ans:
        return None, f"MALFORMED: missing answers.{QUESTION_KEY}.choice"
    probs = ans.get("probabilities")
    if not isinstance(probs, dict) or not probs:
        return None, "MISSING_PROBABILITIES"
    try:
        p = {k: float(probs[k]) for k in LABELS}
    except (KeyError, TypeError, ValueError):
        return None, f"MISSING_OR_BAD_PROBABILITY: {str(probs)[:150]}"
    if any(not (0.0 <= v <= 1.0) for v in p.values()):      # also catches NaN
        return None, f"PROBABILITY_OUT_OF_RANGE: {p}"
    usage = data.get("usage") if isinstance(data.get("usage"), dict) else {}
    tin, tout, cost = num(usage.get("input_tokens")), num(usage.get("output_tokens")), num(usage.get("cost"))
    pE = p["E"]
    pred = 1 if pE >= THRESHOLD else 0
    ad = {k: p[k] for k in ["A", "B", "C", "D"]}
    return {
        "jev_prob_E": pE, "jev_choice": ans.get("choice"),
        "jev_pred_option": "E" if pred == 1 else max(ad, key=ad.get),
        "jev_probabilities": json.dumps(p), "jev_pred": pred,
        "input_tokens": tin, "output_tokens": tout,
        "total_tokens": (tin + tout) if tin is not None and tout is not None else None,
        "cost_usd": cost, "cost_source": "api_reported" if cost is not None else "not_reported",
        "model_served": data.get("model"), "request_id": data.get("id"),
    }, None

def call_jev(payload, stop):
    """One row: up to MAX_ATTEMPTS tries. Returns dict with ok/parsed/error/attempts/aborted."""
    last_err, attempts = "UNKNOWN", 0
    while attempts < MAX_ATTEMPTS:
        if stop.is_set():
            return {"ok": False, "aborted": True, "attempts": attempts, "error": "ABORTED"}
        attempts += 1
        wait = 2 ** attempts
        try:
            r = requests.post(URL, headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
                              json=payload, timeout=REQUEST_TIMEOUT)
        except requests.exceptions.Timeout:
            last_err = f"TIMEOUT after {REQUEST_TIMEOUT}s"
        except requests.exceptions.RequestException as e:
            last_err = f"NETWORK_ERROR: {type(e).__name__}: {str(e)[:150]}"
        else:
            code = r.status_code
            if code == 200:
                try:
                    data = r.json()
                except ValueError:
                    data = None
                    last_err = f"MALFORMED_JSON: {r.text[:150]}"
                if data is not None:
                    parsed, perr = parse_response(data)
                    if parsed:
                        return {"ok": True, "parsed": parsed, "attempts": attempts, "error": None}
                    last_err = perr
            elif code in FATAL_CODES:
                stop.set()
                return {"ok": False, "aborted": False, "attempts": attempts,
                        "error": f"HTTP {code} (fatal, run stopped): {r.text[:200]}"}
            elif code in RETRY_CODES or code >= 500:
                last_err = f"HTTP {code}: {r.text[:150]}"
                if code == 429:
                    try:
                        wait = max(wait, float(r.headers.get("Retry-After", 0)))
                    except ValueError:
                        pass
            else:
                return {"ok": False, "aborted": False, "attempts": attempts,
                        "error": f"HTTP {code} (not retried): {r.text[:200]}"}
        if attempts < MAX_ATTEMPTS:
            time.sleep(wait)
    return {"ok": False, "aborted": False, "attempts": attempts, "error": last_err}

def save_checkpoint():
    tmp = OUTPUT_FILE + ".tmp"
    df.to_csv(tmp, index=False, encoding="utf-8-sig")
    os.replace(tmp, OUTPUT_FILE)

def to_int0(x):
    try:
        return int(float(x))
    except (TypeError, ValueError):
        return 0

def run_rows(indices, label=""):
    indices = list(indices)
    stop, counter = threading.Event(), [0]
    print(f"{label}Scoring {len(indices)} rows with {WORKERS} workers...")

    def record(i, res):
        with lock:
            if res.get("aborted"):
                return                                    # stays PENDING/FAILED, retried next run
            df.at[i, "attempts"] = to_int0(df.at[i, "attempts"]) + res["attempts"]
            if res["ok"]:
                for k, v in res["parsed"].items():
                    df.at[i, k] = v
                df.at[i, "status"], df.at[i, "error"] = "OK", None
            else:
                for k in ["jev_prob_E", "jev_choice", "jev_pred_option", "jev_probabilities", "jev_pred",
                          "input_tokens", "output_tokens", "total_tokens", "cost_usd", "cost_source",
                          "model_served", "request_id"]:
                    df.at[i, k] = None
                df.at[i, "status"], df.at[i, "error"] = "FAILED", res["error"]   # never a prediction of 0
            counter[0] += 1
            if counter[0] % CHECKPOINT_EVERY == 0:
                save_checkpoint()
                n_fail = int((df["status"] == "FAILED").sum())
                print(f"  checkpoint: {counter[0]}/{len(indices)} done this run | FAILED overall: {n_fail}")

    def work(i):
        try:
            payload, ierr = build_payload(df.loc[i])
            res = {"ok": False, "aborted": False, "attempts": 0, "error": ierr} if ierr else call_jev(payload, stop)
        except Exception as e:
            res = {"ok": False, "aborted": False, "attempts": 0, "error": f"INTERNAL: {type(e).__name__}: {e}"}
        record(i, res)

    try:
        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            for f in as_completed([ex.submit(work, i) for i in indices]):
                f.result()
    finally:
        with lock:
            save_checkpoint()                             # also saves on Ctrl-C / stop
    if stop.is_set():
        print("RUN STOPPED EARLY by a fatal API error (key / credits / permission). Check the error column.")

def project_cost():
    """Pilot-based projection of the remaining cost from real usage so far. Returns dict or None."""
    d = df[df["status"] == "OK"].copy()
    d["tin"] = pd.to_numeric(d["input_tokens"], errors="coerce")
    d["cst"] = pd.to_numeric(d["cost_usd"], errors="coerce")
    d["byt"] = payload_bytes.loc[d.index]
    d = d[d["tin"].notna() & (d["byt"] > 0)]
    if d.empty:
        return None
    ratio = d["tin"].sum() / d["byt"].sum()               # input tokens per payload byte
    r_lo, r_hi = (d["tin"] / d["byt"]).min(), (d["tin"] / d["byt"]).max()
    c = d.dropna(subset=["cst"])
    rate = c["cst"].sum() / c["tin"].sum() if len(c) and c["tin"].sum() > 0 else PRICE_IN_PER_M / 1e6
    rem_bytes = payload_bytes.loc[df.index[df["status"] != "OK"]].sum()
    return {"pilot_rows": len(d), "tokens_per_byte": ratio, "rate_per_token": rate,
            "rem_rows": int((df["status"] != "OK").sum()),
            "tok": ratio * rem_bytes, "tok_lo": r_lo * rem_bytes, "tok_hi": r_hi * rem_bytes,
            "cost": ratio * rem_bytes * rate, "cost_lo": r_lo * rem_bytes * rate, "cost_hi": r_hi * rem_bytes * rate,
            "implied_price_per_M": rate * 1e6}

def print_projection():
    p = project_cost()
    if p is None:
        print("No successful pilot rows yet: no reliable estimate. Run the smoke test (cell 5) first.")
        return None
    print("-" * 66)
    print(f"PILOT-BASED PROJECTION (ESTIMATE, from {p['pilot_rows']} scored rows) for {p['rem_rows']} remaining rows")
    print(f"  input tokens/payload byte : {p['tokens_per_byte']:.3f}")
    print(f"  implied API price         : ${p['implied_price_per_M']:.4f} per 1M input tokens "
          f"(published: ${PRICE_IN_PER_M})")
    print(f"  projected input tokens    : ~{p['tok']:,.0f}  (range {p['tok_lo']:,.0f} to {p['tok_hi']:,.0f})")
    print(f"  projected remaining cost  : ~${p['cost']:.4f}  (range ${p['cost_lo']:.4f} to ${p['cost_hi']:.4f})")
    print("  Range = lowest/highest tokens-per-byte seen in the pilot. A small pilot can miss rows that are")
    print("  denser or longer, and failed attempts that are billed would not appear here.")
    print("-" * 66)
    return p


In [6]:
# ==========================================
# 5. SMOKE TEST (first 5 unscored rows) + pilot-based cost projection
# ==========================================
smoke_idx = list(df.index[df["status"] != "OK"][:SMOKE_N])
if not smoke_idx:
    print("Nothing to smoke-test: every row is already OK.")
else:
    print(f"Heuristic worst-case ceiling for these {len(smoke_idx)} rows: "
          f"${payload_bytes.loc[smoke_idx].sum() / 1e6 * PRICE_IN_PER_M:.6f}")
    run_rows(smoke_idx, label="[smoke] ")
    show = ["question", "jev_prob_E", "jev_choice", "jev_pred", "status", "error",
            "attempts", "input_tokens", "output_tokens", "cost_usd"]
    show = [c for c in show if c in df.columns]
    out = df.loc[smoke_idx, show].copy()
    out["question"] = out["question"].astype(str).str.slice(0, 40)
    print(out.to_string())
    if (df.loc[smoke_idx, "status"] == "FAILED").any():
        print("\nSome smoke-test rows FAILED. Read the error column before launching the full run.")
print_projection()


Heuristic worst-case ceiling for these 5 rows: $0.000213
[smoke] Scoring 5 rows with 8 workers...
                                   question jev_prob_E jev_choice jev_pred status error  attempts input_tokens output_tokens  cost_usd
0  মস্তিষ্কের কোন অংশ মানুষের ক্ষুধা নিয়ন্       0.28          C        0     OK  None         1          534            52  0.000022
1       ডেলটয়েড টিউবারসিটি কোন অস্থির অংশ?       0.71          E        1     OK  None         1          558            52  0.000023
2                   ফুসফুসের আবরণকে কি বলে?       0.92          E        1     OK  None         1          496            52  0.000021
3  নিচের কোনটি মানুষের দ্বিতীয় মস্তিষ্ক হি       0.49          E        0     OK  None         1          593            52  0.000025
4   কানের কোন অংশ ভারসাম্য রক্ষায় কাজ করে?       0.65          E        1     OK  None         1          559            52  0.000023
------------------------------------------------------------------
PILOT-BASED PROJECTION (E

{'pilot_rows': 5,
 'tokens_per_byte': np.float64(0.54118111791428),
 'rate_per_token': np.float64(4.2e-08),
 'rem_rows': 989,
 'tok': np.float64(536579.454868655),
 'tok_lo': np.float64(513191.5027777778),
 'tok_hi': np.float64(588961.0922155689),
 'cost': np.float64(0.02253633710448351),
 'cost_lo': np.float64(0.021554043116666668),
 'cost_hi': np.float64(0.024736365873053893),
 'implied_price_per_M': np.float64(0.041999999999999996)}

In [7]:
# ==========================================
# 6. FULL RUN (asks for confirmation; resumes; skips rows already OK)
# ==========================================
todo = df.index[df["status"] != "OK"]
print(f"Rows already scored OK (will NOT be re-run): {int((df['status'] == 'OK').sum())}")
print(f"Rows to run now (PENDING + FAILED)         : {len(todo)}")
proj = print_projection()
if proj is None:
    print(f"Fallback heuristic bounds (unreliable): ${lo_cost:.4f} to ${hi_cost:.4f}")
known = pd.to_numeric(df["cost_usd"], errors="coerce").sum()
print(f"Actual API-reported cost so far: ${known:.6f}")

if len(todo) == 0:
    print("Nothing to run.")
elif input("\nType YES to start the full run (anything else cancels): ").strip() == "YES":
    run_rows(todo, label="[full] ")
    print("Full run finished. Re-run this cell to retry any FAILED rows, then run cell 7.")
else:
    print("Cancelled. No API calls were made.")


Rows already scored OK (will NOT be re-run): 5
Rows to run now (PENDING + FAILED)         : 989
------------------------------------------------------------------
PILOT-BASED PROJECTION (ESTIMATE, from 5 scored rows) for 989 remaining rows
  input tokens/payload byte : 0.541
  implied API price         : $0.0420 per 1M input tokens (published: $0.042)
  projected input tokens    : ~536,579  (range 513,192 to 588,961)
  projected remaining cost  : ~$0.0225  (range $0.0216 to $0.0247)
  Range = lowest/highest tokens-per-byte seen in the pilot. A small pilot can miss rows that are
  denser or longer, and failed attempts that are billed would not appear here.
------------------------------------------------------------------
Actual API-reported cost so far: $0.000115

Type YES to start the full run (anything else cancels): YES
[full] Scoring 989 rows with 8 workers...
  checkpoint: 50/989 done this run | FAILED overall: 0
  checkpoint: 100/989 done this run | FAILED overall: 0
  checkpoint

In [8]:
# ==========================================
# 7. FINAL SUMMARY
# ==========================================
save_checkpoint()
ok = df[df["status"] == "OK"]
scored = len(ok)
e_det = int((pd.to_numeric(ok["jev_pred"]) == 1).sum())
a_d = int((pd.to_numeric(ok["jev_pred"]) == 0).sum())
failed = int((df["status"] == "FAILED").sum())
pending = int((df["status"] == "PENDING").sum())
acc = e_det / scored * 100 if scored else float("nan")

tin = pd.to_numeric(df["input_tokens"], errors="coerce").sum()
tout = pd.to_numeric(df["output_tokens"], errors="coerce").sum()
ttot = pd.to_numeric(df["total_tokens"], errors="coerce").sum()
cost = pd.to_numeric(df["cost_usd"], errors="coerce").sum()
no_cost = int((ok["cost_source"] != "api_reported").sum())
argmax_e = int((ok["jev_choice"] == "E").sum())

print("=" * 66)
print(f"JEV Q-ALT RESULTS | {MODEL} | threshold P(E) >= {THRESHOLD}")
print("=" * 66)
print(f"Accuracy (E detections / scored)  : {acc:.2f}%   ({e_det}/{scored})")
print(f"E detections (label 1)            : {e_det}")
print(f"A-D predictions (label 0)         : {a_d}")
print(f"FAILED (excluded from accuracy)   : {failed}")
print(f"PENDING (excluded from accuracy)  : {pending}")
print(f"Total rows                        : {len(df)}")
print(f"Tokens (API-reported)             : in={int(tin):,}  out={int(tout):,}  total={int(ttot):,}")
print(f"TOTAL KNOWN COST (API-reported)   : ${cost:.6f}")
if no_cost:
    print(f"  note: {no_cost} scored rows had no usage.cost, so the true total may be higher.")
print(f"Diagnostic: rows where Jev's own top choice was E: {argmax_e}/{scored}")
print("Costs of failed or timed-out attempts are not reported by the API and are not included.")
print(f"Saved: {OUTPUT_FILE}")
print("=" * 66)
if failed:
    print("\nFailure reasons:")
    print(df.loc[df["status"] == "FAILED", "error"].astype(str).str.slice(0, 60).value_counts().to_string())
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
except Exception:
    pass


JEV Q-ALT RESULTS | typesafe/jev-1.13 | threshold P(E) >= 0.5
Accuracy (E detections / scored)  : 69.11%   (687/994)
E detections (label 1)            : 687
A-D predictions (label 0)         : 307
FAILED (excluded from accuracy)   : 0
PENDING (excluded from accuracy)  : 0
Total rows                        : 994
Tokens (API-reported)             : in=541,437  out=51,688  total=593,125
TOTAL KNOWN COST (API-reported)   : $0.022740
Diagnostic: rows where Jev's own top choice was E: 744/994
Costs of failed or timed-out attempts are not reported by the API and are not included.
Saved: jev_qalt_evaluations.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>